In [1]:
# Cell 1: Environment Dependencies Installation & System Verification

# Install missing audio dependency and Hugging Face stack while preserving base Colab PyTorch runtime
!pip install -q torchaudio transformers datasets peft bitsandbytes trl accelerate

# Verification: Confirm clean imports of core training modules
import torch
import transformers
from transformers import TrainingArguments
import peft
import bitsandbytes

print("=== Environment Verification ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available. Ensure GPU runtime is enabled in Colab.")
print("TrainingArguments and HF core modules imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 37.8 MB/s eta 0:00:00
=== Environment Verification ===
PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU Device: Tesla T4
TrainingArguments and HF core modules imported successfully!


In [2]:
import os
import torch


#Base model and tokenizer loaders with automatic architecture detection
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)

#PEFT components for low-rank adapter injection and k-bit model stabilization
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#Hugging Face ecosystem data loading and supervised fine tuning trainer
from datasets import load_dataset
from trl import SFTTrainer

#Model choice
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

#Dataset choice
DATASET_ID = "witfoo/precinct6-cybersecurity"
OUTPUT_DIR = "./results/sec-qwen-1.5b-adapter"

#LoRA hyperparameters
LORA_R = 16 #16 provides optimal capacity for domain adaptation without overfitting
LORA_ALPHA = 32 #Scaling factor. Rule is alpha = 2 * r to scale adapter gradients properly
LORA_DROPOUT = 0.05 #5% dropout regularizes training against memorizing raw log string patterns

#Targeting all linear projection layers ensures max expressivity across attention and MLP blocks
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]


In [3]:
#Quantization Setup (4 bit NF4 configuration)

#Configure 4-bit NormalFloat quantization to load base model weights efficiently in VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", #optimal representation for normally distributed weights
    bnb_4bit_compute_dtype=torch.float16, #matches T4 tensor core precision
    bnb_4bit_use_double_quant=True, #Quantizes quantization constants to save extra memory
)

#output quantization settings (verification)
print("=== Quantization Config Initialized ===")
print(f"Load in 4-bit: {bnb_config.load_in_4bit}")
print(f"Quant Type: {bnb_config.bnb_4bit_quant_type}")
print(f"Compute Dtype: {bnb_config.bnb_4bit_compute_dtype}")

=== Quantization Config Initialized ===
Load in 4-bit: True
Quant Type: nf4
Compute Dtype: torch.float16


In [4]:
#Model and tokenizer initialization

#Load tokenizer corresponding to base model architecture
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

#Qwen tokenizer requires explicit padding token definition; fallback to EOS
tokenizer.pad_token = tokenizer.eos_token

#Enforce right padding for autoregressive causal language modeling
tokenizer.padding_side = "right"

#Load base model with 4 bit NormalFloat quantization settings
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto", #Dynamically allocates model layers to available VRAM
    trust_remote_code=True
)

#Freeze base model parameters and cast layer norms for k-bit training stability
model = prepare_model_for_kbit_training(model)

print("=== Model and Tokenizer Initialized ===")
print(f"Model Architecture: {model.config.model_type}")
print(f"VRAM Allocated: {torch.cuda.memory_allocated() / 1e9:2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

=== Model and Tokenizer Initialized ===
Model Architecture: qwen2
VRAM Allocated: 0.957672 GB


In [5]:
#PEFT / LoRA adapter injection setup

#Configure LoRA settings using variables defined in Cell 2
peft_config = LoraConfig(
    r=LORA_R, #Rank dimension (16) controlling adapter matrix capacity
    lora_alpha=LORA_ALPHA, #Scaling factor (32) determining adapter weight impact
    lora_dropout=LORA_DROPOUT, #Dropout probability (0.05) for network regularization
    target_modules=TARGET_MODULES, #injects adapters into all linear layers (q, k, v, o, gate, up, down)
    bias="none", #disables bias parameter updates to restrict training overhead
    task_type="CAUSAL_LM" #Defines causal autoregressive language modeling task
)

#Attach trainable LoRA adapter layers to the 4-bit base model
model = get_peft_model(model, peft_config)

#Print trainable parameter count vs total parameter footprint
print("=== PEFT Adapter Injected ===")
model.print_trainable_parameters()

=== PEFT Adapter Injected ===
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [6]:
#Dataset ingestion and prompt structuring

#load 50.000 logs: Optimal scale for domain adaptation on T4 (google colab) without session timeout
raw_dataset = load_dataset(DATASET_ID, "signals", split="train[:50000]")

def format_instruction_prompt(sample):
    """
    Transforms raw log fields into Qwen's ChatML instruction format.
    """
    return {
        "text": (
            f"<|im_start|>system\nYou are a SOC analyst assistant. Categorize security events accurately.<|im_end|>\n"
            f"<|im_start|>user\nAnalyze log: {sample['message_sanitized']}<|im_end|>\n"
            f"<|im_start|>assistant\nLabel: {sample['label_binary']} | Disposition: {sample['disposition']}<|im_end|>"
        )
    }

  #apply prompt formatting function and drop unneeded raw columns to free system RAM
formatted_dataset = raw_dataset.map(
    format_instruction_prompt,
    remove_columns=raw_dataset.column_names
)

  #output: single formatted training sample
print("=== Dataset Formatted Succesfully ===")
print(f"Total Processed Samples: {len(formatted_dataset)}")
print("\n--- Sample Prompt Output ---")
print(formatted_dataset[0]["text"])

README.md: 0.00B [00:00, ?B/s]

signals/signals.parquet:   0%|          | 0.00/429M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

=== Dataset Formatted Succesfully ===
Total Processed Samples: 50000

--- Sample Prompt Output ---
<|im_start|>system
You are a SOC analyst assistant. Categorize security events accurately.<|im_end|>
<|im_start|>user
Analyze log: <185>Jun 28 08:28:20 USER-0015-1367 CEF:0|Barracuda|WAF|1220|342|GEO_IP_BLOCK|1|cat=WF dst=10.68.218.61 dpt=443 act=DENY msg=[GeoIP Policy Match] duser="-" src=100.64.95.151 spt=47220 reHOST-0210Method=GET app=TLSv1.2 reHOST-0210Context="-" start=Jun 28 2024 08:28:20  rt=1719581300278  reHOST-0210=/wp-content/plugins/wp-central/readme.txt reHOST-0210ClientApplication="Mozilla/5.0 (ORG-0362 NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.113 Safari/537.36 " dvcUSER-0015=HOST-1367 externalId=1905f085636-697b5667 cn2=47220 cn2Label=ProxyPort cs1=global cs1Label=RuleID cs2=NONE cs2Label=FollowUpAction cs3=GLOBAL cs3Label=RuleType cs4=Forceful Browsing cs4Label=AttackGroup cs5=100.64.95.151 cs5Label=ProxyIP cs6="-" cs6Label=SessionID 
<

In [7]:
#Training engine configuration

#Configure execution parameters for low-bit supervised fine-tuning
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    #Micro-batch size 4 + accumulation. 4 = Effective Batch Size of 16
    #Keeps VRAM consumption under 8GB on T4/P100 GPUs while maintaining stable gradient updates
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    #2e-4 is the empirical standard learning rate for QLoRA adapters
    learning_rate=2e-4,

    #log loss every 10 steps for clean monitoring without output clutter
    logging_steps=10,

    #1 epoch over 50k records prevents network overfitting on raw string patterns
    num_train_epochs=1,

    #enable FP16 mixed precision execution matching T4 Tensor cores
    fp16=True,

    #paged_adamw_8bit pages memory to host RAM if VRAM spikes occur
    optim="paged_adamw_8bit",

    save_strategy="epoch",
    report_to="none", #supresses external metric logging (like WandB)
)

#output: active training configuration settings
print("=== Training Arguments Initialized ===")
print(f"Effective Batch Size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Optimizer: {training_args.optim}")
print(f"FP16 Enabled: {training_args.fp16}")

=== Training Arguments Initialized ===
Effective Batch Size: 16
Optimizer: OptimizerNames.PAGED_ADAMW_8BIT
FP16 Enabled: True


In [8]:
# Cell 8: Trainer Initialization with Learning Rate Decay and Step Checkpoints

import torch
import trl.trainer.sft_trainer
from trl import SFTConfig, SFTTrainer
from transformers import Trainer
import os
# Força o uso de apenas 1 GPU para evitar o conflito do DataParallel com bitsandbytes
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Convert all trainable LoRA parameters to float32
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# Bypass TRL chunking patches
trl.trainer.sft_trainer._patch_chunked_ce_lm_head = lambda *args, **kwargs: None
SFTTrainer.compute_loss = Trainer.compute_loss

# Define SFT configuration with LR decay and frequent checkpoints
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,             # Reduced LR to eliminate gradient spikes after step 1200
    lr_scheduler_type="cosine",     # Cosine decay smoothly reduces LR as training progresses
    max_grad_norm=0.3,              # Strict gradient clipping
    warmup_steps=150,               # Warmup for initial stability
    logging_steps=10,
    num_train_epochs=1,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    save_strategy="steps",          # Save intermediate checkpoints to prevent losing progress
    save_steps=300,
    save_total_limit=2,             # Keeps only the 2 best/latest checkpoints to save disk space
    report_to="none"
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    args=sft_config
)

# Execute the fine-tuning loop
print("=== Starting QLoRA Fine-Tuning Loop (Optimized Run) ===")
trainer.train()
print("\n=== Training Completed Successfully ===")

Adding EOS to train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/50000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


=== Starting QLoRA Fine-Tuning Loop (Optimized Run) ===


Step,Training Loss
10,3.296540
20,3.195531
30,3.007794
40,2.712189
50,2.343756
60,1.905535
70,1.409977
80,1.041737
90,0.745998
100,0.796554



=== Training Completed Successfully ===


In [9]:
# Cell 9: Save Fine-Tuned LoRA Adapter and Tokenizer with Disk Verification

import os

OUTPUT_DIR = "/kaggle/working/qwen_lora_output"

print(f"=== SALVANDO ADAPTADOR FINETUNADO EM: {OUTPUT_DIR} ===")

# 1. Cria o diretório de saída
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Salva o modelo do adaptador LoRA
try:
    if 'trainer' in globals() and hasattr(trainer, 'model'):
        trainer.model.save_pretrained(OUTPUT_DIR)
        print("✅ Pesos do adaptador salvos via 'trainer.model.save_pretrained'.")
    elif 'model' in globals():
        model.save_pretrained(OUTPUT_DIR)
        print("✅ Pesos do adaptador salvos via 'model.save_pretrained'.")
    else:
        print("⚠️ Objeto 'trainer' ou 'model' não encontrado no escopo global.")
except Exception as e:
    print(f"❌ Erro ao salvar os pesos do modelo: {e}")

# 3. Salva o Tokenizador
try:
    if 'tokenizer' in globals():
        tokenizer.save_pretrained(OUTPUT_DIR)
        print("✅ Tokenizador salvo com sucesso.")
    else:
        print("⚠️ Objeto 'tokenizer' não encontrado no escopo global.")
except Exception as e:
    print(f"❌ Erro ao salvar o tokenizador: {e}")

# 4. Verificação física dos arquivos no disco
print("\n=== CONFERÊNCIA DOS ARQUIVOS SALVOS NO DISCO ===")
if os.path.exists(OUTPUT_DIR):
    saved_files = os.listdir(OUTPUT_DIR)
    if saved_files:
        print(f"Arquivos localizados na pasta '{OUTPUT_DIR}':")
        for file_name in sorted(saved_files):
            file_path = os.path.join(OUTPUT_DIR, file_name)
            if os.path.isfile(file_path):
                size_mb = os.path.getsize(file_path) / (1024 * 1024)
                print(f"  ├── {file_name} ({size_mb:.2f} MB)")
            elif os.path.isdir(file_path):
                print(f"  ├── [Diretório] {file_name}")
        print("\n✅ Salvamento finalizado e verificado com sucesso!")
    else:
        print("❌ Alerta: A pasta de saída foi criada, mas está vazia.")
else:
    print("❌ Alerta: Diretório de saída não encontrado.")

=== Checking Saved Checkpoints on Disk ===
Found saved model folders/checkpoints:
 - ./results/sec-qwen-1.5b-adapter
     ├── README.md (0.00 MB)
 - ./results/sec-qwen-1.5b-adapter/checkpoint-3000
     ├── scheduler.pt (0.00 MB)
     ├── tokenizer.json (10.89 MB)
     ├── trainer_state.json (0.05 MB)
     ├── README.md (0.00 MB)
     ├── training_args.bin (0.01 MB)
     ├── adapter_model.safetensors (35.27 MB)
     ├── optimizer.pt (36.22 MB)
     ├── chat_template.jinja (0.00 MB)
     ├── tokenizer_config.json (0.00 MB)
     ├── adapter_config.json (0.00 MB)
     ├── rng_state.pth (0.01 MB)
 - ./results/sec-qwen-1.5b-adapter/checkpoint-3125
     ├── scheduler.pt (0.00 MB)
     ├── tokenizer.json (10.89 MB)
     ├── trainer_state.json (0.05 MB)
     ├── README.md (0.00 MB)
     ├── training_args.bin (0.01 MB)
     ├── adapter_model.safetensors (35.27 MB)
     ├── optimizer.pt (36.22 MB)
     ├── chat_template.jinja (0.00 MB)
     ├── tokenizer_config.json (0.00 MB)
     ├── adapter_con

In [10]:
import os
import shutil

source_dir = "./results/sec-qwen-1.5b-adapter/checkpoint-3125"
target_dir = "/kaggle/working/modelo_final"

# Copia a pasta com os arquivos finais do LoRA
shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)

print(f"=== ARQUIVOS COPIADOS COM SUCESSO PARA: {target_dir} ===")
for f in sorted(os.listdir(target_dir)):
    fpath = os.path.join(target_dir, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f" ├── {f} ({size_mb:.2f} MB)")

=== ARQUIVOS COPIADOS COM SUCESSO PARA: /kaggle/working/modelo_final ===
 ├── README.md (0.00 MB)
 ├── adapter_config.json (0.00 MB)
 ├── adapter_model.safetensors (35.27 MB)
 ├── chat_template.jinja (0.00 MB)
 ├── optimizer.pt (36.22 MB)
 ├── rng_state.pth (0.01 MB)
 ├── scheduler.pt (0.00 MB)
 ├── tokenizer.json (10.89 MB)
 ├── tokenizer_config.json (0.00 MB)
 ├── trainer_state.json (0.05 MB)
 ├── training_args.bin (0.01 MB)


In [16]:
# Cell 10: Test Inference with Fine-Tuned Model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Caminho onde os pesos estão salvos
LORA_PATH = "/kaggle/working/modelo_final"
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Modelo base de 1.5B utilizado

print("=== 1. Carregando Tokenizer e Modelo Base ===")
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("=== 2. Acoplando o Adaptador LoRA Treinado ===")
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

print("\n=== Modelo pronto! Gerando resposta de teste... ===")

# Teste com uma pergunta do seu domínio/dataset
prompt = "Explique como funciona a arquitetura de um modelo de linguagem finetunado com LoRA."
messages = [{"role": "user", "content": prompt}]
text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(text_input, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,  # Aumentado para dar espaço ao texto completo
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("\n--------------------------------------------------")
print("RESPOSTA DO SEU MODELO FINETUNADO:")
print("--------------------------------------------------")
print(response)

=== 1. Carregando Tokenizer e Modelo Base ===


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== 2. Acoplando o Adaptador LoRA Treinado ===

=== Modelo pronto! Gerando resposta de teste... ===

--------------------------------------------------
RESPOSTA DO SEU MODELO FINETUNADO:
--------------------------------------------------
Aqui está uma explicação sobre a arquitetura de um modelo de linguagem finetuned com LoRA:

1) O que é Lora (Low-Rank Adaptation):
LoRA é um algoritmo que permite ajustar rapidamente o tamanho da matriz de pesos do modelo. Isso significa que ele pode reduzir o número de parâmetros sem comprometer significativamente a precisão.

2) O que é a arquitetura:
Para modelos de linguagem finetuned com LoRA, a estrutura geral é similar ao original. A principal diferença reside na implementação da "adaptação" dos pesos, baseada no método LoRA.

3) Como funciona:
O processo envolve três etapas principais: 
- Pré-processamento da sequência de texto.
- Treinamento básico com assegurar a consistência e o padronização das sequências de entrada.
- Execução de treino ad